<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "360"
iso_country = "Indonesia"
year = 2025
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 1.450*1e12

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Indonesia': 

Number of trade records present in dataset: 55209
Number of partner countries present in dataset: 194
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$205875.0m
Total weight within the dataset: 231071.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,360,Indonesia,699,India,Import,Sea,847751,847751 | Machinery; for mould,8477,8477 | Machinery; for worki,84,84 | Machinery and mechan,44915,Machinery n.e.c. for working rubber or plastic...,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,854.000,NaN,854.000,NaN,0.0,NaN,56.705600,5100.0,3168.992100,179.699598,"['Container', 'Vehicle']","['Container', 'Vehicle']",0.684150,0.682021,0.682217,0.682217,0.683399,0.683483,0.717185,1.316016,0.784302,0.729586,0.732716,0.752964,0.701175,1.264807,0.823110,0.758293,0.740804,0.777265,0.687245,1.397364,1.024353,0.900148,0.832202,0.811293,0.673539,1.485918,1.198729,1.117045,0.886055,0.871415,0.660190,1.512969,1.438590,1.464483,0.953857,0.916006,0.646401,1.475128,1.406730,1.515630,1.144290,0.932743,0.633353,1.303609,1.360232,1.446652,1.072589,0.943620,0.620256,1.185556,1.211153,1.427163,1.166314,0.952942,44915
1,1,360,Indonesia,040,Austria,Import,Sea,340540,340540 | Scouring pastes and,3405,"3405 | Polishes, creams, sc",34,"34 | Soap, organic surfac",35334,Scouring pastes and powders and other scouring...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",226.541,NaN,226.541,NaN,0.0,NaN,58.356962,11576.0,7192.990696,419.761082,['Container'],['Container'],1.496271,1.491800,1.492178,1.492178,1.494740,1.494747,1.571041,2.903862,1.724784,1.602860,1.608585,1.651584,1.535674,2.796832,1.812792,1.665564,1.625565,1.706666,1.504666,3.091136,2.260009,1.978414,1.826309,1.782778,1.474163,3.270035,2.645852,2.461748,1.951883,1.916801,1.444405,3.353815,3.197898,3.233644,2.100150,2.016483,1.413628,3.285397,3.144117,3.375528,2.561271,2.054293,1.384499,2.897368,3.016345,3.193015,2.364522,2.079073,1.355262,2.621947,2.675511,3.153346,2.598836,2.094782,35334


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $23108.5m (11.2%), ... 

                               cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                               
10 | Cereals               3.747550e+09  1.326951e+10      1.8      5.7
23 | Food industries, res  3.342907e+09  7.252494e+09      1.6      3.1
17 | Sugars and sugar con  2.360956e+09  4.668351e+09      1.1      2.0
18 | Cocoa and cocoa prep  2.149670e+09  7.999844e+08      1.0      0.3
08 | Fruit and nuts, edib  2.005242e+09  1.611955e+09      1.0      0.7
04 | Dairy produce; birds  1.623288e+09  6.283979e+08      0.8      0.3
12 | Oil seeds and oleagi  1.602696e+09  2.920328e+09      0.8      1.3
21 | Miscellaneous edible  1.164920e+09  2.596600e+08      0.6      0.1
02 | Meat and edible meat  1.041691e+09  2.696947e+08      0.5      0.1
07 | Vegetables and certa  9.686955e+08  1.403281e+09      0.5      0.6
01 | Animals; live         6.0971

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $15937.5m (7.7%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Food & Beverages  9.462151e+09  9.956829e+09      4.6      4.3
Agriculture       6.360438e+09  5.298166e+09      3.1      2.3
Fishing           1.148982e+08  2.625479e+07      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $16358.6m (7.9%), ... 

                                   cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                  
OFD - Food products n.e.c.     4.175254e+09  2.786589e+09      2.0      1.2
OCR - Crops n.e.c.             3.114515e+09  1.004970e+09      1.5      0.4
VOL - Vegetable oils and fats  2.426371e+09  5.228453e+09      1.2      2.3
V_F - Vegetables, fruit, nuts  2.203720e+09  2.258947e+09      1.1      1.0
MIL - Dairy products           1.655567e+09  6.695683e+08      0.8      0.3
CMT - Bovine meat prods        1.010858e+09  2.574659e+08      0.5      0.1
PFB - Plant-based fibers       7.104293e+08  3.677042e+08      0.3      0.2
OMT - Meat products n.e.c.     3.599119e+08  3.316800e+08      0.2      0.1
GRO - Cereal grains n.e.c.     2.329551e+08  9.542061e+08      0.1      0.4
PCR - Processed rice           2.171581e+08  3.772037e+08      0.1      0.2
OAP - Animal products

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 11.2% 
GTAP 7.9% 
EORA 7.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27831/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,360,Indonesia,699,India,Import,Sea,847751,847751 | Machinery; for mould,8477,8477 | Machinery; for worki,84,84 | Machinery and mechan,44915,Machinery n.e.c. for working rubber or plastic...,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,854.000,NaN,854.000,NaN,0.0,NaN,56.705600,5100.0,3168.992100,179.699598,"['Container', 'Vehicle']","['Container', 'Vehicle']",0.684150,0.682021,0.682217,0.682217,0.683399,0.683483,0.717185,1.316016,0.784302,0.729586,0.732716,0.752964,0.701175,1.264807,0.823110,0.758293,0.740804,0.777265,0.687245,1.397364,1.024353,0.900148,0.832202,0.811293,0.673539,1.485918,1.198729,1.117045,0.886055,0.871415,0.660190,1.512969,1.438590,1.464483,0.953857,0.916006,0.646401,1.475128,1.406730,1.515630,1.144290,0.932743,0.633353,1.303609,1.360232,1.446652,1.072589,0.943620,0.620256,1.185556,1.211153,1.427163,1.166314,0.952942,44915,NaN
1,1,360,Indonesia,040,Austria,Import,Sea,340540,340540 | Scouring pastes and,3405,"3405 | Polishes, creams, sc",34,"34 | Soap, organic surfac",35334,Scouring pastes and powders and other scouring...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",226.541,NaN,226.541,NaN,0.0,NaN,58.356962,11576.0,7192.990696,419.761082,['Container'],['Container'],1.496271,1.491800,1.492178,1.492178,1.494740,1.494747,1.571041,2.903862,1.724784,1.602860,1.608585,1.651584,1.535674,2.796832,1.812792,1.665564,1.625565,1.706666,1.504666,3.091136,2.260009,1.978414,1.826309,1.782778,1.474163,3.270035,2.645852,2.461748,1.951883,1.916801,1.444405,3.353815,3.197898,3.233644,2.100150,2.016483,1.413628,3.285397,3.144117,3.375528,2.561271,2.054293,1.384499,2.897368,3.016345,3.193015,2.364522,2.079073,1.355262,2.621947,2.675511,3.153346,2.598836,2.094782,35334,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $13211.9m of $205875.0m (6.4%)
Volume: 23143.1 of 231071.1 million kilograms (10.0%)
Value-to-Weight: 0.57 from 0.89 US$ per kg (+-35.9%)
Average distance to market (volume-weighted): 6068.5 from 4716.0 nm (+28.7%)

            food_group  N_com      cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     39  3.529907e+09  1.289147e+10      1.7      5.6
1      Sugar and Honey    153  2.363932e+09  4.669204e+09      1.1      2.0
2               Fruits    167  1.999160e+09  1.609231e+09      1.0      0.7
3    Processed Cereals    722  1.994390e+09  1.312133e+09      1.0      0.6
4                Dairy    147  1.592520e+09  6.181946e+08      0.8      0.3
5           Vegetables    302  1.021092e+09  1.436739e+09      0.5      0.6
6        Meat and Prep     54  4.223487e+08  3.955360e+08      0.2      0.2
7        Fats and Oils    258  2.885797e+08  2.105551e+08      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.591384e+09,2.933570e+09,1.745865e+09,1.623034e+09,1.628788e+09,1.672490e+09,1.463844e+09,3.387199e+09,3.229537e+09,3.266849e+09,2.124586e+09,2.040228e+09,1.374009e+09,2.650941e+09,2.705882e+09,3.187736e+09,2.627025e+09,2.119334e+09
1,Fats and Oils,2.434560e+08,4.252298e+08,2.635658e+08,2.461758e+08,2.469551e+08,2.544785e+08,2.260103e+08,4.880570e+08,4.655636e+08,4.750372e+08,3.189362e+08,3.047267e+08,2.137474e+08,3.898605e+08,3.993878e+08,4.672774e+08,3.858450e+08,3.154366e+08
2,Fruits,1.965648e+09,3.573523e+09,2.148134e+09,2.000790e+09,2.007750e+09,2.062814e+09,1.812480e+09,4.139260e+09,3.943042e+09,3.998182e+09,2.616182e+09,2.504493e+09,1.704546e+09,3.250034e+09,3.318782e+09,3.892646e+09,3.214052e+09,2.599192e+09
3,Meat and Prep,1.195207e+09,2.209182e+09,1.312171e+09,1.219414e+09,1.223770e+09,1.256482e+09,1.098865e+09,2.551495e+09,2.432877e+09,2.460072e+09,1.597739e+09,1.534087e+09,1.031048e+09,1.994709e+09,2.035459e+09,2.398983e+09,1.977127e+09,1.593655e+09
4,Processed Cereals,1.483536e+09,2.649180e+09,1.612797e+09,1.505199e+09,1.510570e+09,1.553881e+09,1.372086e+09,3.077491e+09,2.933920e+09,2.983981e+09,1.972137e+09,1.873592e+09,1.293657e+09,2.432547e+09,2.487882e+09,2.899520e+09,2.402191e+09,1.941175e+09
5,Sugar and Honey,6.323465e+09,1.011399e+10,6.669542e+09,6.302467e+09,6.326621e+09,6.550465e+09,5.950914e+09,1.196759e+10,1.143731e+10,1.185658e+10,8.308298e+09,7.587296e+09,5.690474e+09,9.849284e+09,1.015783e+10,1.139604e+10,9.619463e+09,7.786853e+09
6,Unprocessed Cereals,3.245104e+10,5.950918e+10,3.554471e+10,3.306350e+10,3.318214e+10,3.408553e+10,2.987642e+10,6.881820e+10,6.562685e+10,6.644794e+10,4.333802e+10,4.149303e+10,2.806462e+10,5.394869e+10,5.508303e+10,6.474614e+10,5.342976e+10,4.307554e+10
7,Vegetables,1.962990e+09,3.620408e+09,2.153775e+09,2.002117e+09,2.009247e+09,2.063145e+09,1.805464e+09,4.183020e+09,3.988078e+09,4.034133e+09,2.622594e+09,2.517104e+09,1.694553e+09,3.271939e+09,3.339116e+09,3.933061e+09,3.242256e+09,2.614497e+09


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.84,0.10,0.02,0.02,0.05,1.21,1.11,1.13,0.41,0.36,0.80,0.84,1.14,0.79,0.47
1,Fats and Oils,0.63,0.07,0.01,0.01,0.04,0.91,0.83,0.86,0.32,0.27,0.61,0.64,0.88,0.60,0.35
2,Fruits,0.80,0.09,0.02,0.02,0.05,1.16,1.07,1.09,0.40,0.35,0.77,0.81,1.09,0.76,0.45
3,Meat and Prep,2.40,0.28,0.06,0.07,0.15,3.44,3.16,3.22,1.18,1.03,2.28,2.38,3.24,2.24,1.33
4,Processed Cereals,0.58,0.06,0.01,0.01,0.04,0.86,0.78,0.81,0.30,0.25,0.57,0.60,0.81,0.56,0.32
5,Sugar and Honey,1.60,0.15,-0.01,0.00,0.10,2.55,2.32,2.50,1.00,0.69,1.76,1.89,2.41,1.66,0.89
6,Unprocessed Cereals,7.67,0.88,0.17,0.21,0.46,11.03,10.13,10.36,3.81,3.29,7.33,7.65,10.39,7.19,4.25
7,Vegetables,1.62,0.19,0.04,0.05,0.10,2.33,2.14,2.18,0.80,0.70,1.54,1.61,2.19,1.52,0.90
